<!-- # Install OpenAI (Run only once)

!pip install -q openai -->

In [1]:
# # Install OpenAI (Run only once)

# !pip install -q openai

In [2]:
import os 
import json 
import sqlite3 
from openai import OpenAI

In [3]:
# Create OpenAI Client

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
MODEL = "gpt-4o-mini"

In [5]:
# SQlite db connection
connection = sqlite3.connect("hospital.db")
cursor = connection.cursor()

In [6]:

## create patients table

cursor.execute(""" 
CREATE TABLE IF NOT EXISTS patients (
               patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
               name TEXT,
               age INTEGER,
               gender TEXT,
               phone TEXT
               )""")

connection.commit()

## create appointments table
# Create Appointments Table

cursor.execute("""
CREATE TABLE IF NOT EXISTS appointments (

    appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,

    patient_id INTEGER,

    doctor TEXT,

    appointment_date TEXT,

    FOREIGN KEY(patient_id)
    REFERENCES patients(patient_id)

)
""")

connection.commit()


In [7]:
# Check Existing Tables

cursor.execute("""

SELECT name
FROM sqlite_master
WHERE type='table'

""")

print(cursor.fetchall())

[('patients',), ('sqlite_sequence',), ('appointments',)]


In [9]:
def add_patient(name, age, gender, phone):
    cursor.execute(
        """ 
INSERT INTO patients (name, age, gender, phone)
VALUES (?, ?, ?, ?)""",
(name, age, gender, phone)
    )
    connection.commit()

    return "Patient added successfully"

# Search Patient

def search_patient(name):

    cursor.execute(
        """
        SELECT *
        FROM patients
        WHERE name = ?
        """,
        (name,)
    )

    patient = cursor.fetchall()

    if patient:
        return str(patient)

    return "Patient not found."

# Update Patient Phone Number

def update_patient(patient_id, phone):

    cursor.execute(
        """
        UPDATE patients
        SET phone = ?
        WHERE patient_id = ?
        """,
        (phone, patient_id)
    )

    connection.commit()

    if cursor.rowcount == 0:
        return "Patient not found."

    return "Patient updated successfully."

# Delete Patient

def delete_patient(patient_id):

    cursor.execute(
        """
        DELETE FROM patients
        WHERE patient_id = ?
        """,
        (patient_id,)
    )

    connection.commit()

    if cursor.rowcount == 0:
        return "Patient not found."

    return "Patient deleted successfully."

In [10]:
# Book Appointment

def book_appointment(patient_id, doctor, appointment_date):

    cursor.execute(
        """
        INSERT INTO appointments
        (patient_id, doctor, appointment_date)

        VALUES (?, ?, ?)
        """,
        (patient_id, doctor, appointment_date)
    )

    connection.commit()

    return "Appointment booked successfully."

# View Appointments

def view_appointments(patient_id):

    cursor.execute(
        """
        SELECT *
        FROM appointments
        WHERE patient_id = ?
        """,
        (patient_id,)
    )

    appointments = cursor.fetchall()

    if appointments:
        return str(appointments)

    return "No appointments found."

# Cancel Appointment

def cancel_appointment(appointment_id):

    cursor.execute(
        """
        DELETE FROM appointments
        WHERE appointment_id = ?
        """,
        (appointment_id,)
    )

    connection.commit()

    if cursor.rowcount == 0:
        return "Appointment not found."

    return "Appointment cancelled successfully."

In [11]:
print(add_patient("John", 28, "Male", "9876543210"))

print(add_patient("Alice", 34, "Female", "9999999999"))

print(search_patient("John"))

print(update_patient(1, "8888888888"))

print(search_patient("John"))

print(book_appointment(1, "Dr. Smith", "2026-07-05"))

print(view_appointments(1))

print(cancel_appointment(1))

print(view_appointments(1))

Patient added successfully
Patient added successfully
[(1, 'John', 28, 'Male', '9876543210')]
Patient updated successfully.
[(1, 'John', 28, 'Male', '8888888888')]
Appointment booked successfully.
[(1, 1, 'Dr. Smith', '2026-07-05')]
Appointment cancelled successfully.
No appointments found.


In [1]:
# Tool Definitions

tools = [

    {
        "type": "function",
        "function": {
            "name": "add_patient",
            "description": "Register a new patient.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string"
                    },
                    "age": {
                        "type": "integer"
                    },
                    "gender": {
                        "type": "string"
                    },
                    "phone": {
                        "type": "string"
                    }
                },
                "required": [
                    "name",
                    "age",
                    "gender",
                    "phone"
                ]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "search_patient",
            "description": "Search a patient using the patient's name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string"
                    }
                },
                "required": [
                    "name"
                ]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "update_patient",
            "description": "Update the phone number of a patient.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "integer"
                    },
                    "phone": {
                        "type": "string"
                    }
                },
                "required": [
                    "patient_id",
                    "phone"
                ]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "delete_patient",
            "description": "Delete a patient using the patient ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "integer"
                    }
                },
                "required": [
                    "patient_id"
                ]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "book_appointment",
            "description": "Book a doctor's appointment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "integer"
                    },
                    "doctor": {
                        "type": "string"
                    },
                    "appointment_date": {
                        "type": "string"
                    }
                },
                "required": [
                    "patient_id",
                    "doctor",
                    "appointment_date"
                ]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "view_appointments",
            "description": "View all appointments of a patient.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {
                        "type": "integer"
                    }
                },
                "required": [
                    "patient_id"
                ]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "cancel_appointment",
            "description": "Cancel an appointment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "appointment_id": {
                        "type": "integer"
                    }
                },
                "required": [
                    "appointment_id"
                ]
            }
        }
    }

]

In [2]:
# Tool Mapping

available_functions = {

    "add_patient": add_patient,

    "search_patient": search_patient,

    "update_patient": update_patient,

    "delete_patient": delete_patient,

    "book_appointment": book_appointment,

    "view_appointments": view_appointments,

    "cancel_appointment": cancel_appointment

}

NameError: name 'add_patient' is not defined

In [3]:
# Conversation History

messages = [

    {
        "role": "system",
        "content": """
You are an AI Hospital Receptionist.

Your responsibilities include:

- Registering new patients.
- Searching patient records.
- Updating patient information.
- Deleting patients.
- Booking appointments.
- Viewing appointments.
- Cancelling appointments.

Whenever a user's request requires database information,
use the available tools.

After receiving a tool result,
analyze it and provide a helpful response to the user.
"""
    }

]

In [4]:
# Hospital Agent

def hospital_agent():

    while True:

        user_input = input("\nYou : ")

        if user_input.lower() == "exit":
            print("\nAssistant : Goodbye!")
            break

        messages.append(
            {
                "role": "user",
                "content": user_input
            }
        )

        while True:

            response = client.chat.completions.create(

                model=MODEL,

                messages=messages,

                tools=tools,

                tool_choice="auto"

            )

            assistant_message = response.choices[0].message

            messages.append(assistant_message)

            # Check whether the model wants to call a tool
            if assistant_message.tool_calls:

                for tool_call in assistant_message.tool_calls:

                    function_name = tool_call.function.name

                    function_arguments = json.loads(
                        tool_call.function.arguments
                    )

                    selected_function = available_functions[function_name]

                    tool_result = selected_function(**function_arguments)

                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": str(tool_result)
                        }
                    )

                # Continue the loop so the LLM can observe
                # the tool output and decide what to say next.
                continue

            else:

                print("\nAssistant :", assistant_message.content)

                break

In [ ]:
hospital_agent()